# PHASE 3 - Time Series Split

This notebook splits the processed feature table in chronological order into 70% train, 15% validation, and 15% test. No shuffle and no model training are used.

The split is made after feature engineering. Lag and rolling features were generated from past rows only, so future target values are not used as predictors.

In [4]:
from pathlib import Path

import pandas as pd

processed_candidates = [
    Path("data/processed/hour_features.csv"),
    Path("../data/processed/hour_features.csv"),
]
processed_path = next((path for path in processed_candidates if path.exists()), None)
if processed_path is None:
    raise FileNotFoundError("Run PHASE 2 first to create data/processed/hour_features.csv.")

df = pd.read_csv(processed_path, parse_dates=["timestamp", "dteday"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded: {processed_path}")
print(f"Rows: {len(df)}")
print(f"Chronological order: {df['timestamp'].is_monotonic_increasing}")
df.head()

Loaded: data\processed\hour_features.csv
Rows: 17211
Chronological order: True


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,...,day_of_year,is_weekend,is_workingday,rush_hour,lag_1,lag_2,lag_24,lag_168,rolling_mean_24,rolling_mean_168
0,169,2011-01-08,1,0,1,7,0,6,0,2,...,8,1,0,1,2.0,5.0,84.0,16.0,63.208333,56.458333
1,170,2011-01-08,1,0,1,8,0,6,0,3,...,8,1,0,1,9.0,2.0,210.0,40.0,60.083333,56.416667
2,171,2011-01-08,1,0,1,9,0,6,0,3,...,8,1,0,1,15.0,9.0,134.0,32.0,51.958333,56.267857
3,172,2011-01-08,1,0,1,10,0,6,0,2,...,8,1,0,0,20.0,15.0,63.0,13.0,47.208333,56.196429
4,173,2011-01-08,1,0,1,11,0,6,0,2,...,8,1,0,0,61.0,20.0,67.0,1.0,47.125000,56.482143


## Chronological split

In [5]:
train_end = int(len(df) * 0.70)
validation_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end].copy()
validation_df = df.iloc[train_end:validation_end].copy()
test_df = df.iloc[validation_end:].copy()

assert len(train_df) + len(validation_df) + len(test_df) == len(df)
assert train_df["timestamp"].max() < validation_df["timestamp"].min()
assert validation_df["timestamp"].max() < test_df["timestamp"].min()

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(validation_df), len(test_df)],
    "share": [len(train_df) / len(df), len(validation_df) / len(df), len(test_df) / len(df)],
    "first_timestamp": [train_df["timestamp"].min(), validation_df["timestamp"].min(), test_df["timestamp"].min()],
    "last_timestamp": [train_df["timestamp"].max(), validation_df["timestamp"].max(), test_df["timestamp"].max()],
})
display(split_summary)
print("Chronological split checks passed.")

,split,rows,share,first_timestamp,last_timestamp
0,train,12047,0.699959,2011-01-08 07:00:00,2012-05-29 03:00:00
1,validation,2582,0.150020,2012-05-29 04:00:00,2012-09-13 17:00:00
2,test,2582,0.150020,2012-09-13 18:00:00,2012-12-31 23:00:00


Chronological split checks passed.


## Save split datasets

These files contain the historical features and target for each split. Later model notebooks must fit preprocessing steps only on `train_df` and use validation/test only for evaluation.

In [ ]:
split_dir = processed_path.parent
split_dir.mkdir(parents=True, exist_ok=True)

train_path = split_dir / "train.csv"
validation_path = split_dir / "validation.csv"
test_path = split_dir / "test.csv"

train_df.to_csv(train_path, index=False)
validation_df.to_csv(validation_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Saved train: {train_path}")
print(f"Saved validation: {validation_path}")
print(f"Saved test: {test_path}")

Saved train: data\processed\train.csv
Saved validation: data\processed\validation.csv
Saved test: data\processed\test.csv


## Phase 3 conclusion

The data is now separated chronologically. PHASE 4 can train and evaluate Linear Regression using the saved splits.